In [1]:
# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time # To time execution

# Data and Preprocessing
import yfinance as yf
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller

# ARIMA
# Ensure pmdarima is installed: pip install pmdarima
import pmdarima as pm # For auto_arima

# Visualization
import plotly.graph_objects as go

# Plotting Style Preferences (Optional)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

In [2]:
# ## 2. Configuration

# %%
# --- User Defined Parameters ---
ticker = "BTC-USD"
start_date = "2017-11-09"
end_date = "2025-01-01"

# >>> New Parameter <<<
forecast_horizon = 30 # Predict 30 days ahead

# Define Train/Test Split Ratio for the *initial* training phase
train_split_ratio = 0.80

# auto_arima parameters
ARIMA_SEASONAL_PERIOD = 7 # Assuming weekly seasonality for daily data might be relevant
# Set ARIMA_SEASONAL_PERIOD to 1 if no seasonality is assumed

# Optional: Refitting frequency (0 = fit once initially, >0 = refit every N steps)
# Refitting is computationally expensive for ARIMA but can capture parameter drift.
# Keeping it off (0) for this example, matching the original code's structure.
REFIT_FREQUENCY = 0

In [3]:
# ## 3. Data Loading and Preparation

# %%
print(f"--- Loading Data for {ticker} ---")
try:
    df_full = yf.download(tickers=[ticker], start=start_date, end=end_date, progress=False)
    if df_full.empty:
        raise ValueError(f"No data downloaded for {ticker}.")

    # Select 'Close' column directly
    if 'Close' not in df_full.columns:
         raise ValueError(f"'Close' column not found in downloaded data for {ticker}.")
    df_full = df_full[['Close']].copy()

    # Ensure daily frequency and fill missing values
    df_full = df_full.asfreq('D')
    df_full.ffill(inplace=True) # Forward fill potential gaps (e.g., weekends)
    df_full.dropna(inplace=True) # Drop any NaNs remaining (e.g., at the very start)
    if df_full.empty:
        raise ValueError(f"Data for {ticker} became empty after processing.")
    print(f"Loaded {len(df_full)} data points for {ticker} from {df_full.index.min().strftime('%Y-%m-%d')} to {df_full.index.max().strftime('%Y-%m-%d')}.")
except Exception as e:
    print(f"Original Error: {e}")
    # Raising ValueError for consistency in error handling checks
    raise ValueError(f"Failed to load or process data for {ticker}. Check symbol and data source.")

--- Loading Data for BTC-USD ---
YF.download() has changed argument auto_adjust default to True
Loaded 2610 data points for BTC-USD from 2017-11-09 to 2024-12-31.


In [4]:
# ## 4. Data Splitting

# %%
# Split Data into initial training and test sets
n_total = len(df_full)
n_train = int(train_split_ratio * n_total)
n_test = n_total - n_train # n_test determines the number of walk-forward steps

train_data_df = df_full[:n_train]
test_data_df = df_full[n_train:] # Holds the actual values observed during the test period

print(f"\nInitial Training Data: {n_train} points ({train_data_df.index.min().strftime('%Y-%m-%d')} to {train_data_df.index.max().strftime('%Y-%m-%d')})")
# Note: The test period starts here, but evaluation is on t+30 actuals
print(f"Test Period Start Date: {test_data_df.index.min().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps: {n_test}")


Initial Training Data: 2088 points (2017-11-09 to 2023-07-28)
Test Period Start Date: 2023-07-29
Number of Walk-Forward Steps: 522


In [5]:
# ## 5. Stationarity Check (on Initial Training Data)

# %%
def check_stationarity(timeseries, ts_name="Time Series"):
    """Performs ADF test and prints results."""
    print(f"\n--- Stationarity Check Results of Dickey-Fuller Test for {ts_name} ---")
    # Drop NA values to prevent errors in adfuller
    timeseries_clean = timeseries.dropna()
    if timeseries_clean.empty:
        print("Skipping test: Time series is empty after dropping NA.")
        return
    dftest = adfuller(timeseries_clean, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=["Test Statistic", "p-value", "#Lags Used", "# Observations Used"],
    )
    for key, value in dftest[4].items():
        dfoutput[f"Critical Value ({key})"] = value
    print(dfoutput.to_string()) # Use to_string for clean print
    if dftest[1] <= 0.05:
        print("=> Conclusion: Data is likely Stationary (reject H0)")
    else:
        print("=> Conclusion: Data is likely Non-Stationary (fail to reject H0)")

# Check stationarity on the 'Close' price of the initial training data
check_stationarity(train_data_df['Close'], ts_name=f"{ticker} Initial Training 'Close'")


--- Stationarity Check Results of Dickey-Fuller Test for BTC-USD Initial Training 'Close' ---
Test Statistic            -1.459481
p-value                    0.553452
#Lags Used                24.000000
# Observations Used     2063.000000
Critical Value (1%)       -3.433524
Critical Value (5%)       -2.862942
Critical Value (10%)      -2.567516
=> Conclusion: Data is likely Non-Stationary (fail to reject H0)


In [6]:
# ## 6. Initial ARIMA Model Fit (using auto_arima)

# %%
print("\n--- Fitting Initial ARIMA Model on Training Data ---")
start_time_initial_train = time.time()

# Use auto_arima to find the best model ONCE on the initial training data
# This determines the (p,d,q)(P,D,Q)m order
# `d` and `D` will be determined automatically based on stationarity tests ('adf', 'kpss', 'pp')
arima_model = pm.auto_arima(train_data_df['Close'],
                           start_p=1, start_q=1,       # Start searching from simple models
                           test='adf',                # Use ADF test to determine 'd'
                           max_p=3, max_q=3,          # Max non-seasonal orders
                           m=ARIMA_SEASONAL_PERIOD,   # Seasonal period
                           start_P=0,                 # Start searching from P=0 for seasonal
                           seasonal=(ARIMA_SEASONAL_PERIOD > 1), # Enable seasonal search if m > 1
                           d=None,                    # Let ADF test determine 'd'
                           D=None,                    # Let seasonal test (e.g., Canova-Hansen) determine 'D'
                           trace=False,               # Don't print internal steps
                           error_action='ignore',     # Don't stop if a model fails to fit
                           suppress_warnings=True,    # Don't show convergence warnings etc.
                           stepwise=True)             # Use efficient stepwise algorithm

end_time_initial_train = time.time()
print(f"Initial ARIMA training finished in {end_time_initial_train - start_time_initial_train:.2f} seconds.")
print("\n--- Initial Best Model Found ---")
# Print the summary which includes model order, coefficients, etc.
print(arima_model.summary())
# Explicitly print orders for clarity
print(f"\nBest ARIMA Order (p,d,q): {arima_model.order}")
if arima_model.seasonal_order:
    print(f"Best Seasonal Order (P,D,Q,m): {arima_model.seasonal_order}")
else:
    print("Seasonal Order: Not Applicable (m=1 or seasonality disabled)")


--- Fitting Initial ARIMA Model on Training Data ---
Initial ARIMA training finished in 225.90 seconds.

--- Initial Best Model Found ---
                                      SARIMAX Results                                      
Dep. Variable:                                   y   No. Observations:                 2088
Model:             SARIMAX(0, 1, 0)x(0, 0, [1], 7)   Log Likelihood              -17323.491
Date:                             Wed, 16 Apr 2025   AIC                          34650.982
Time:                                     13:47:14   BIC                          34662.269
Sample:                                 11-09-2017   HQIC                         34655.118
                                      - 07-28-2023                                         
Covariance Type:                               opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------

In [7]:
# ## 7. Walk-Forward Validation (Rolling Forecast) Loop

# %%
print(f"\n--- Starting ARIMA Walk-Forward Validation for {n_test} steps (t+{forecast_horizon}) ---")
start_time_walk_forward = time.time()

arima_walk_forward_predictions = [] # List to store the t+horizon predictions

# Make a copy of the initially trained model to perform updates on
current_arima_model = arima_model

for t in range(n_test):
    # 1. Predict 'forecast_horizon' steps ahead from the current state of the model
    # `predict` returns an array of forecasts from t+1 to t+forecast_horizon
    try:
        predictions_h_steps = current_arima_model.predict(n_periods=forecast_horizon, return_conf_int=False)
    except Exception as e:
        print(f"Error during prediction at step {t+1}: {e}. Appending NaN.")
        predictions_h_steps = np.full(forecast_horizon, np.nan) # Handle prediction failure

    # 2. Extract the specific forecast for the target horizon (t+forecast_horizon)
    # This is the last element of the predictions array
    yhat_h = predictions_h_steps[-1] if len(predictions_h_steps) == forecast_horizon else np.nan
    arima_walk_forward_predictions.append(yhat_h)

    # 3. Get the *actual* value for the *next step* (t+1) to update the model state
    # This value corresponds to index `t` in the `test_data_df`
    # Ensure we don't try to access beyond the test data length
    if t < len(test_data_df):
        actual_value_t_plus_1 = test_data_df['Close'].iloc[t]

        # 4. Update the model with the single actual observation for time t+1.
        # This efficiently updates the model's internal state without a full refit,
        # preparing it for the *next* prediction cycle starting from t+1.
        try:
            # Check if actual_value_t_plus_1 is NaN before updating
            if pd.notna(actual_value_t_plus_1):
                 current_arima_model.update(actual_value_t_plus_1)
            else:
                 print(f"Warning: Skipping update at step {t+1} due to NaN actual value.")
        except Exception as e:
            print(f"Warning: ARIMA update failed at step {t+1} with error: {e}. Model state might be stale.")
            # Depending on error severity, consider breaking or attempting a refit if configured
    else:
        # This case should ideally not be reached if n_test is calculated correctly
        print(f"Warning: Reached end of test data ({t}) before completing all {n_test} steps.")
        break


    # --- Optional: Periodic Refitting Point ---
    # Refitting is computationally expensive but can adapt to changing dynamics better over long horizons.
    if REFIT_FREQUENCY > 0 and (t + 1) % REFIT_FREQUENCY == 0 and (t + 1) < n_test:
        print(f"\n--- Refitting ARIMA at step {t+1}/{n_test} ---")
        refit_start_time = time.time()
        # Define the history up to the point *before* the current prediction cycle started
        current_history = df_full['Close'].iloc[:n_train + t + 1] # Data available *after* observing step t's actual
        try:
            current_arima_model = pm.auto_arima(current_history,
                                          start_p=1, start_q=1, test='adf',
                                          max_p=3, max_q=3, m=ARIMA_SEASONAL_PERIOD,
                                          start_P=0, seasonal=(ARIMA_SEASONAL_PERIOD > 1),
                                          d=None, D=None, trace=False,
                                          error_action='ignore', suppress_warnings=True,
                                          stepwise=True,
                                          # Optional: Start search from previous best params? (can speed up)
                                          # start_params=current_arima_model.params()
                                          )
            refit_end_time = time.time()
            print(f"ARIMA Refitting complete in {refit_end_time - refit_start_time:.2f} seconds.")
            print(f"New Best Order: {current_arima_model.order}, Seasonal: {current_arima_model.seasonal_order}")
        except Exception as e:
            print(f"Error during refitting at step {t+1}: {e}. Continuing with previous model.")
            # Continue using the model state from before the failed refit attempt
    # --- End Refitting ---


    # Log progress periodically
    if (t + 1) % 100 == 0:
        print(f"ARIMA Walk-Forward Step {t+1}/{n_test} complete.")


end_time_walk_forward = time.time()
total_walk_forward_time = end_time_walk_forward - start_time_walk_forward
print(f"\nARIMA Walk-Forward finished in {total_walk_forward_time:.2f} seconds.")

# Ensure predictions list is numpy array for metrics
arima_walk_forward_predictions = np.array(arima_walk_forward_predictions)



--- Starting ARIMA Walk-Forward Validation for 522 steps (t+30) ---
ARIMA Walk-Forward Step 100/522 complete.
ARIMA Walk-Forward Step 200/522 complete.
ARIMA Walk-Forward Step 300/522 complete.
ARIMA Walk-Forward Step 400/522 complete.
ARIMA Walk-Forward Step 500/522 complete.

ARIMA Walk-Forward finished in 3.48 seconds.


In [8]:
# ## 8. Evaluate Walk-Forward Performance

# %%
# Define the evaluation metrics function
def evaluate_forecast(y_true, y_pred, model_name, horizon):
    """Calculates and prints standard evaluation metrics, handling potential NaNs."""
    # Remove NaN values resulting from prediction/update errors or alignment
    valid_indices = ~np.isnan(y_pred) & ~np.isnan(y_true)
    y_true_clean = y_true[valid_indices].flatten()
    y_pred_clean = y_pred[valid_indices].flatten()

    num_eval_points = len(y_true_clean)
    print(f"\n--- {model_name} Walk-Forward (t+{horizon}) Evaluation Results ---")
    print(f"Number of valid evaluation points: {num_eval_points}")

    if num_eval_points == 0:
        print("Evaluation skipped: No valid overlapping points found.")
        return {'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'R2': np.nan}

    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    # MAPE calculation needs care for potential zeros in actuals
    # Avoid division by zero if actual value is zero
    non_zero_actuals = y_true_clean != 0
    if np.any(non_zero_actuals):
         mape = np.mean(np.abs((y_true_clean[non_zero_actuals] - y_pred_clean[non_zero_actuals]) / y_true_clean[non_zero_actuals])) * 100
    else:
         mape = np.nan # Undefined if all actuals are zero

    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    try:
        # R2 score requires at least two data points
        r2 = r2_score(y_true_clean, y_pred_clean) if num_eval_points >= 2 else np.nan
    except ValueError: r2 = np.nan # Should not happen with >=2 points, but safe practice

    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"MAPE: {mape:.4f}%") # Display MAPE as percentage
    print(f"R²:   {r2:.4f}")
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# --- Correct Alignment for Evaluation ---
# Predictions made during the loop correspond to future dates.
# The prediction made at step `t` (using data up to `n_train + t`)
# forecasts the value for date `df_full.index[n_train + t + forecast_horizon]`

# Determine the indices of the actual values corresponding to the predictions
start_actual_idx = n_train + forecast_horizon
# The last prediction corresponds to the actual value at index n_train + n_test -1 + forecast_horizon
end_actual_idx = n_train + n_test + forecast_horizon # Target index is EXCLUSIVE for slicing

# Ensure the end index does not exceed the total data length
max_actual_idx = len(df_full)
end_actual_idx = min(end_actual_idx, max_actual_idx)

# Adjust start if it exceeds data length (can happen if test set is very short)
start_actual_idx = min(start_actual_idx, max_actual_idx)


# Extract the actual values for comparison
y_test_actual = df_full['Close'].values[start_actual_idx : end_actual_idx]

# Trim predictions: we only need as many predictions as actual values available for the horizon
num_actual_for_eval = len(y_test_actual)
# The number of predictions made is n_test. We need to select the ones corresponding to y_test_actual.
# The first prediction corresponds to y_test_actual[0].
# The last prediction corresponds to y_test_actual[-1].
# We need predictions from index 0 up to num_actual_for_eval.
arima_walk_forward_predictions_eval = arima_walk_forward_predictions[:num_actual_for_eval]

print(f"\n--- Evaluation Alignment ---")
print(f"Number of predictions made: {len(arima_walk_forward_predictions)}")
print(f"Actual data start index for eval: {start_actual_idx}")
print(f"Actual data end index for eval (exclusive): {end_actual_idx}")
print(f"Number of actual data points for eval: {len(y_test_actual)}")
print(f"Number of predictions used for eval: {len(arima_walk_forward_predictions_eval)}")


# Check for length mismatch *after* alignment
if len(y_test_actual) != len(arima_walk_forward_predictions_eval):
     # This check should ideally not fail if logic above is correct, but good failsafe
    raise ValueError(f"Length mismatch after alignment: Actual ({len(y_test_actual)}) vs Predictions ({len(arima_walk_forward_predictions_eval)})")


# Evaluate against the correctly aligned actual test data
arima_wf_results = evaluate_forecast(y_test_actual, arima_walk_forward_predictions_eval, f"ARIMA ({ticker})", forecast_horizon)


--- Evaluation Alignment ---
Number of predictions made: 522
Actual data start index for eval: 2118
Actual data end index for eval (exclusive): 2610
Number of actual data points for eval: 492
Number of predictions used for eval: 492


IndexError: boolean index did not match indexed array along dimension 1; dimension is 1 but corresponding boolean dimension is 492

In [ ]:
# ## 9. Visualize Walk-Forward Results

# %%
print("\n--- Plotting Walk-Forward Forecasts ---")

# Get the correct dates for the x-axis (corresponding to the target forecast dates)
actual_dates = df_full.index[start_actual_idx : end_actual_idx]

# Ensure date length matches evaluation data length
if len(actual_dates) != len(arima_walk_forward_predictions_eval):
     # If evaluation resulted in zero points, actual_dates might be empty
     if len(arima_walk_forward_predictions_eval) == 0 and len(actual_dates) == 0:
         print("Skipping plot: No evaluation points.")
     else:
        raise ValueError(f"Date length ({len(actual_dates)}) mismatch with evaluated predictions ({len(arima_walk_forward_predictions_eval)}) for plotting.")

# Create DataFrame for plotting only if there's data to plot
if len(actual_dates) > 0:
    results_df_wf = pd.DataFrame({
        'Actual': y_test_actual.flatten(), # Use the aligned actuals
        f'ARIMA (t+{forecast_horizon})': arima_walk_forward_predictions_eval.flatten() # Use the aligned predictions
    }, index=actual_dates) # Use the target dates as index

    fig = go.Figure()
    # Plot actual data for the evaluation period
    fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf['Actual'], mode='lines', name='Actual Price (Eval Period)', line=dict(color='black')))
    # Plot the t+30 forecasts
    fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf[f'ARIMA (t+{forecast_horizon})'], mode='lines', name=f'ARIMA Walk-Forward (t+{forecast_horizon})', line=dict(color='blue', dash='dash'))) # Changed color

    fig.update_layout(
        title=f'ARIMA Walk-Forward (t+{forecast_horizon}) Forecast Comparison for {ticker}',
        xaxis_title="Date (Target Date of Forecast)",
        yaxis_title="Price (USD)",
        legend_title="Data/Model",
        template="plotly_white"
    )
    fig.show()
else:
    print("Plot skipped as there are no valid evaluation points.")


In [ ]:
# ## 10. Walk-Forward Evaluation Period Summary

# %%
print(f"\n--- Walk-Forward Evaluation Summary (t+{forecast_horizon}) ---")
print(f"Initial Training Data End Date: {train_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps Made: {n_test}")
print(f"Number of Forecasts Evaluated: {len(arima_walk_forward_predictions_eval)}") # Use length after alignment
if len(arima_walk_forward_predictions_eval) > 0 and 'actual_dates' in locals() and len(actual_dates) > 0:
    print(f"Evaluation Period (Target Dates): {actual_dates.min().strftime('%Y-%m-%d')} to {actual_dates.max().strftime('%Y-%m-%d')}")
else:
    print("Evaluation Period: N/A (No points evaluated or plotted)")
print(f"Forecast Horizon: {forecast_horizon} days")